# Chapter 17 — FlashAttention and the Memory Wall

Two chapters have now ended at the same wall from opposite directions.

Chapter 15 measured attention's peak memory growing **4× for every doubling** of
sequence length, and built a state space model to avoid the quadratic term
entirely, at the cost of lossy recall. Chapter 16 shrank every parameter tensor
in a fine-tuning run by two orders of magnitude, and then noted that none of it
touched the largest *activation*, which is the $n \times n$ attention matrix.

This chapter attacks that tensor directly, and the result is strange enough to
state up front:

> **FlashAttention does not reduce attention's asymptotic cost.** It is still
> $O(n^2)$ arithmetic. It computes exactly the same function, to the same
> precision, with no approximation anywhere. It is simply *never written down*,
> and that alone is worth two orders of magnitude of memory and a large multiple
> of speed.

The reason that works is the most transferable idea in this chapter: on a modern
GPU, attention is not limited by arithmetic. It is limited by **moving bytes**,
and the $n \times n$ matrix is almost entirely traffic.

| Module | What you build | Task |
|---|---|---|
| 1 | Where attention's memory actually goes: counted, then measured | — |
| 2 | The **IO model**: HBM vs SRAM, and attention's arithmetic intensity | — |
| 3 | **Online softmax**: a one-pass softmax with a running max, verified exact | — |
| 4 | **FlashAttention's forward pass** in pure PyTorch, verified against torch's kernel | — |
| 5 | The race: naive vs tiled vs fused, in memory and wall-clock | — |

**How each concept is presented**, the same three passes as earlier chapters:

> 🧠 **The intuition:** the idea in plain language, no symbols.
> 📐 **The math:** the same idea written precisely, so you can read papers.
> 💻 **The code:** the same idea again, executable, in the cell that follows.

**Runtime:** under a minute on a GPU. Nothing is trained and nothing downloads:
this chapter is arithmetic, one algorithm, and a benchmark. Knobs are marked
`# <- knob`. A CUDA device is required for the memory measurements.


In [ ]:
import math, time
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
plt.rcParams["figure.figsize"] = (7, 4.5)
print("torch", torch.__version__, "| device:", device)
if device == "cuda":
    p = torch.cuda.get_device_properties(0)
    print(f"{p.name} | {p.total_memory / 2**30:.0f} GB HBM | {p.multi_processor_count} SMs")

---
# Module 1 — Where the Memory Goes

## 1.1 Count it before measuring it

🧠 **The intuition.** Write out attention as a sequence of statements and ask
what each one leaves in memory:

```
S = Q @ K.T / sqrt(d)      # (B, H, n, n)   <- allocated
P = softmax(S, dim=-1)     # (B, H, n, n)   <- allocated
O = P @ V                  # (B, H, n, d)
```

The inputs $Q, K, V$ are each $B H n d$, **linear** in sequence length. The two
intermediates are $B H n^2$, **quadratic**. And it is worse than it looks,
because autograd has to keep $P$ alive until the backward pass, so that tensor
is not transient: it occupies memory for the entire forward pass of every
remaining layer.

📐 **The arithmetic.** With batch $B$, heads $H$, length $n$, head width $d$, in
fp32:

$$ \underbrace{3 B H n d \cdot 4}_{Q, K, V} \quad\text{vs}\quad \underbrace{2 B H n^2 \cdot 4}_{S,\, P} $$

The crossover is at $n = \tfrac{3}{2} d$, around 96 tokens for $d = 64$. Past
that, essentially all of attention's memory is a matrix you never wanted.

💻 **The code.** Compute the ratio, then check it against the allocator.


In [ ]:
B, H, D = 2, 4, 64                                   # <- knobs
print(f"batch {B}, heads {H}, head dim {D}, fp32\n")
print(f"{'n':>7}{'Q,K,V (MB)':>13}{'S,P (MB)':>12}{'ratio':>9}")
for n in (128, 512, 2048, 8192):
    qkv = 3 * B * H * n * D * 4 / 2**20
    sp = 2 * B * H * n * n * 4 / 2**20
    print(f"{n:>7}{qkv:>13.2f}{sp:>12.2f}{sp / qkv:>8.0f}x")


def naive_attention(Q, K, V):
    '''Attention exactly as Chapter 9 wrote it, and exactly as it should not be run.'''
    S = (Q @ K.transpose(-2, -1)) / math.sqrt(Q.shape[-1])
    return F.softmax(S, dim=-1) @ V


if device == "cuda":
    n = 4096
    Q, K, V = (torch.randn(B, H, n, D, device=device) for _ in range(3))
    torch.cuda.reset_peak_memory_stats()
    baseline = torch.cuda.memory_allocated() / 2**20
    with torch.no_grad():
        naive_attention(Q, K, V)
    print(f"\nmeasured peak at n={n}: {torch.cuda.max_memory_allocated() / 2**20 - baseline:.1f} MB")
    print(f"predicted (S + P)      : {2 * B * H * n * n * 4 / 2**20:.1f} MB")

---
# Module 2 — The IO Model

## 2.1 Why arithmetic is not the bottleneck

🧠 **The intuition.** A GPU has two kinds of memory, and they are not remotely
alike:

| | Size | Bandwidth |
|---|---|---|
| **HBM** (the capacity printed on the box) | tens of GB | ~1–3 TB/s |
| **SRAM** (on-chip, per streaming multiprocessor) | ~100–200 KB | ~10–20 TB/s |

An operation reads from HBM, computes in SRAM, writes back to HBM. If it does
very little arithmetic per byte moved, the arithmetic units sit idle waiting for
memory, and the operation runs at a speed set entirely by bandwidth. That is
called being **memory-bound**, and it is the normal condition for most deep
learning kernels.

📐 **Arithmetic intensity** is the ratio that decides this: FLOPs performed per
byte moved. Look at attention's pieces separately, per head, per batch element:

| Step | FLOPs | Bytes moved | Intensity |
|---|---|---|---|
| $QK^\top$ | $2n^2 d$ | $\sim 2nd \cdot 4 + n^2 \cdot 4$ | high-ish |
| **softmax** | $\sim 5n^2$ | $2 n^2 \cdot 4$ | $\approx 0.6$ |
| $PV$ | $2n^2 d$ | $n^2 \cdot 4 + \dots$ | high-ish |

The softmax is the disaster. It reads an $n \times n$ matrix, does a handful of
operations per element, and writes an $n \times n$ matrix back. Modern GPUs need
an intensity in the **hundreds** to saturate their arithmetic units; softmax
offers well under one.

So the middle of attention is pure memory traffic, and the traffic exists only
because we chose to write $S$ down.

💻 **The code.** Measure the effective bandwidth to see the claim directly.

In [ ]:
if device == "cuda":
    n = 4096
    S = torch.randn(B, H, n, n, device=device)
    elems = S.numel()

    def timeit(fn, reps=20):
        with torch.no_grad():
            for _ in range(3):
                fn()
            torch.cuda.synchronize()
            t0 = time.time()
            for _ in range(reps):
                fn()
            torch.cuda.synchronize()
        return (time.time() - t0) / reps

    t_softmax = timeit(lambda: F.softmax(S, dim=-1))
    t_matmul = timeit(lambda: Q @ K.transpose(-2, -1))

    bytes_softmax = 2 * elems * 4                       # read S, write P
    flops_matmul = 2 * B * H * n * n * D

    print(f"softmax over a {tuple(S.shape)} tensor")
    print(f"  time            : {t_softmax * 1e3:.2f} ms")
    print(f"  bytes moved     : {bytes_softmax / 2**30:.2f} GB")
    print(f"  effective BW    : {bytes_softmax / t_softmax / 1e12:.2f} TB/s   <- bandwidth-limited")
    print(f"  arithmetic done : {5 * elems / t_softmax / 1e12:.2f} TFLOP/s")
    print(f"\nQ @ K.T on the same shapes")
    print(f"  time            : {t_matmul * 1e3:.2f} ms")
    print(f"  arithmetic done : {flops_matmul / t_matmul / 1e12:.1f} TFLOP/s   <- compute-limited")

The two numbers are the whole argument. The matmul runs at a large fraction of
the card's peak arithmetic throughput; the softmax runs at a small fraction of
it while saturating memory bandwidth. Attention spends much of its time in the
operation that does the least useful work.

**So the goal is not fewer FLOPs. It is fewer round trips.** If $S$ never left
SRAM, the softmax would cost essentially nothing, and the only thing standing
in the way is that softmax appears to need the whole row before it can produce
any output.


---
# Module 3 — Online Softmax

🧠 **The intuition.** Softmax looks inherently two-pass. You need the maximum of
the row for numerical stability, and the sum of the exponentials to normalize,
and both are properties of the *entire* row, which seems to force you to have
the whole row in hand.

The way out is to keep a **running** max and a **running** sum, and to *correct*
them when a later element turns out to be bigger than anything seen so far. If
the max increases from $m$ to $m'$, every exponential computed so far was scaled
by the wrong constant, but uniformly so, by exactly $e^{m - m'}$. One
multiplication fixes all of them at once.

📐 **The math.** Process the row in chunks. After chunk $j$, keep

$$ m_j = \max(m_{j-1}, \max x^{(j)}), \qquad
\ell_j = e^{m_{j-1} - m_j}\,\ell_{j-1} + \textstyle\sum_i e^{x^{(j)}_i - m_j} $$

The factor $e^{m_{j-1} - m_j}$ is the retroactive correction. It is $\le 1$
always, so nothing overflows, and the result is **algebraically identical** to
the two-pass version, not an approximation and not a bound.

💻 **The code.** The scalar version first, because it is four lines and the whole
idea is in them. Compare against `F.softmax` on deliberately large values, where
a naive one-pass implementation would overflow.


In [ ]:
def online_softmax(x):
    '''One pass, running max and running sum, with retroactive rescaling.'''
    m = torch.tensor(float("-inf"))
    l = torch.tensor(0.0)
    for xi in x:                                  # streaming, one element at a time
        m_new = torch.maximum(m, xi)
        l = torch.exp(m - m_new) * l + torch.exp(xi - m_new)   # <- the correction
        m = m_new
    return torch.exp(x - m) / l


x = torch.randn(64) * 5                           # wide range: exp() would overflow naively
print(f"max |online - F.softmax| : {(online_softmax(x) - F.softmax(x, 0)).abs().max().item():.3e}")
print(f"largest input            : {x.max().item():.2f}   (exp of this alone = {math.exp(x.max().item()):.3e})")
print("\nnever needed the whole row at once, only a running (m, l) pair.")

---
# Module 4 — FlashAttention

🧠 **The intuition.** Online softmax removes the reason to materialize a full
row. Now put it inside a tiling loop:

- Cut $Q$ into blocks of rows and $K, V$ into blocks of columns.
- For one $Q$ block, walk over the $K/V$ blocks. Each step computes a small
  $b_q \times b_k$ score tile, small enough to live in SRAM, applies online
  softmax to it, and accumulates into a running output.
- When the walk finishes, that block of $O$ is final. Nothing $n \times n$ was
  ever written.

The output accumulator needs the same retroactive correction as $\ell$: when the
running max changes, the partial output computed under the old max is rescaled
by the same $e^{m - m'}$.

📐 **The math.** For query block $i$, iterating over key blocks $j$:

$$ m^{(j)} = \max\big(m^{(j-1)},\, \text{rowmax}(S_{ij})\big), \qquad
\alpha = e^{m^{(j-1)} - m^{(j)}} $$

$$ \ell^{(j)} = \alpha\, \ell^{(j-1)} + \textstyle\sum \exp\big(S_{ij} - m^{(j)}\big), \qquad
O^{(j)} = \alpha\, O^{(j-1)} + \exp\big(S_{ij} - m^{(j)}\big) V_j $$

and finally $O_i = O^{(J)} / \ell^{(J)}$. Peak intermediate memory is
$O(b_q b_k)$ instead of $O(n^2)$, and it is independent of $n$.

**The backward pass gets the same treatment**, which is the other half of the
paper: rather than storing $P$ for the backward pass, store only $(m, \ell)$,
two vectors of length $n$, and **recompute** the score tiles when the gradient
needs them. Recomputation is cheap; the round trip to HBM is not. That trade
(spend FLOPs to avoid bytes) runs against every instinct trained on
computational-complexity courses, and it is the correct instinct on this
hardware.

💻 **The code.** Two loops and the running state. Compare against torch's own
fused kernel, which *is* FlashAttention.


In [ ]:
def flash_attention(Q, K, V, bq=128, bk=128):        # <- knobs: the tile sizes
    '''FlashAttention forward, in pure PyTorch. No n x n tensor is ever allocated.'''
    B, H, N, D = Q.shape
    scale = D ** -0.5
    O = torch.zeros_like(Q)
    for i in range(0, N, bq):                        # over blocks of queries
        Qi = Q[:, :, i:i + bq]
        Oi = torch.zeros_like(Qi)
        mi = torch.full((B, H, Qi.shape[2], 1), float("-inf"), device=Q.device, dtype=Q.dtype)
        li = torch.zeros((B, H, Qi.shape[2], 1), device=Q.device, dtype=Q.dtype)
        for j in range(0, N, bk):                    # over blocks of keys/values
            Kj, Vj = K[:, :, j:j + bk], V[:, :, j:j + bk]
            S = (Qi @ Kj.transpose(-2, -1)) * scale  # (B, H, bq, bk)  <- the only score tensor
            m_new = torch.maximum(mi, S.amax(-1, keepdim=True))
            correction = torch.exp(mi - m_new)       # rescales everything accumulated so far
            P = torch.exp(S - m_new)
            li = correction * li + P.sum(-1, keepdim=True)
            Oi = correction * Oi + P @ Vj            # the output accumulator is corrected too
            mi = m_new
        O[:, :, i:i + bq] = Oi / li                  # normalize once, at the end
    return O


N = 512
Q, K, V = (torch.randn(B, H, N, D, device=device) for _ in range(3))
ref = F.scaled_dot_product_attention(Q, K, V)        # torch's fused FlashAttention kernel

print(f"max |ours  - torch fused| : {(flash_attention(Q, K, V) - ref).abs().max().item():.3e}")
print(f"max |naive - torch fused| : {(naive_attention(Q, K, V) - ref).abs().max().item():.3e}")
print(f"\nlargest score tensor allocated: {(B, H, 128, 128)} instead of {(B, H, N, N)}"
      f"  ({N * N / (128 * 128):.0f}x smaller)")

Three implementations, one function, agreement at the level of float32 rounding.
That is the claim worth internalizing: **FlashAttention is exact.** It is not
sparse attention, not low-rank attention, not a linear-attention approximation.
Those all change what is computed; this changes only where it is stored.

---
# Module 5 — The Race

💻 **The code.** Peak memory and wall-clock for all three, over increasing
sequence lengths. The naive version will run out of memory eventually, which is
itself the result.

In [ ]:
def peak_mb(fn):
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    base = torch.cuda.memory_allocated()
    with torch.no_grad():
        fn()
    return (torch.cuda.max_memory_allocated() - base) / 2**20


def ms(fn, reps=10):
    with torch.no_grad():
        for _ in range(3):
            fn()
        torch.cuda.synchronize()
        t0 = time.time()
        for _ in range(reps):
            fn()
        torch.cuda.synchronize()
    return (time.time() - t0) / reps * 1e3


LENGTHS = (512, 1024, 2048, 4096, 8192)              # <- knob
res = {k: {"mb": [], "ms": []} for k in ("naive", "tiled", "fused")}
print(f"{'n':>6}{'naive MB':>11}{'tiled MB':>11}{'fused MB':>11}"
      f"{'naive ms':>11}{'tiled ms':>11}{'fused ms':>11}")
for n in LENGTHS:
    Q, K, V = (torch.randn(B, H, n, D, device=device) for _ in range(3))
    fns = {"naive": lambda: naive_attention(Q, K, V),
           "tiled": lambda: flash_attention(Q, K, V),
           "fused": lambda: F.scaled_dot_product_attention(Q, K, V)}
    for name, fn in fns.items():
        try:
            res[name]["mb"].append(peak_mb(fn)); res[name]["ms"].append(ms(fn))
        except torch.OutOfMemoryError:
            res[name]["mb"].append(float("nan")); res[name]["ms"].append(float("nan"))
            torch.cuda.empty_cache()
    print(f"{n:>6}" + "".join(f"{res[k]['mb'][-1]:>11.1f}" for k in res)
          + "".join(f"{res[k]['ms'][-1]:>11.2f}" for k in res), flush=True)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
style = {"naive": ("o", "materialized (Chapter 9's code)"),
         "tiled": ("s", "tiled, pure PyTorch"),
         "fused": ("^", "torch fused kernel")}
for key, ylabel, title in [("mb", "peak memory (MB)", "Memory: quadratic vs constant"),
                           ("ms", "time (ms)", "Wall-clock: the kernel is the point")]:
    ax = axes[0] if key == "mb" else axes[1]
    for name, (mk, label) in style.items():
        ax.plot(LENGTHS, res[name][key], marker=mk, label=label)
    ax.set_xscale("log", base=2); ax.set_yscale("log", base=2)
    ax.set_xlabel("sequence length n"); ax.set_ylabel(ylabel); ax.set_title(title)
    ax.grid(True, which="both", alpha=0.3); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

g = lambda k: res[k]["mb"][-1] / res[k]["mb"][-2]
print(f"memory growth, n={LENGTHS[-2]} -> {LENGTHS[-1]} (a 2x increase):")
print(f"  materialized : {g('naive'):.2f}x   <- 4x, quadratic")
print(f"  tiled        : {g('tiled'):.2f}x   <- 2x, and only from Q/K/V themselves")
print(f"\nat n={LENGTHS[-1]}: materialized needs {res['naive']['mb'][-1] / res['fused']['mb'][-1]:.0f}x "
      f"the memory of the fused kernel, for the same answer")

## 5.1 Reading the race

**Memory behaves exactly as designed.** The materialized version grows 4× per
doubling; both tiled versions grow roughly 2×, and that residual growth is $Q$,
$K$ and $V$ themselves, since the score matrix contributes nothing to the trend,
because it never exists. By $n = 8192$ the gap is more than two orders of
magnitude, and our pure-PyTorch tiling lands within a small factor of the real
kernel's footprint. The algorithm, not the CUDA, is what buys the memory.

**Wall-clock tells a different story, and it is the important one.** Our tiled
implementation is *dramatically slower* than both alternatives, despite doing the
same arithmetic as the naive one and asymptotically less memory traffic. The
reason is that we did not actually implement FlashAttention. We implemented its
*schedule* in Python. Every one of the $(n/b_q) \times (n/b_k)$ inner iterations
launches a handful of separate CUDA kernels, each of which dutifully round-trips
its tile through HBM. We reorganized the computation to be SRAM-friendly and
then denied it any way to stay in SRAM.

The real kernel fuses that entire inner loop into one launch, so the tile is
loaded once, the softmax and the accumulation happen in registers and shared
memory, and only the final output block is written. **Same algorithm, ~100×
different wall-clock.**

This is the third time this curriculum has landed on the same point from a
different direction (Chapter 15's scan, Chapter 16's 4-bit matmul, and now
this), so it is worth stating as a rule:

> An algorithm's cost on paper and its cost on a GPU are different quantities.
> The gap between them is memory traffic, and closing it is what a kernel is for.


---
# Wrap-Up

| You built | The transferable lesson |
|---|---|
| The memory count | Attention's inputs are linear in $n$; its intermediates are quadratic. Past $n \approx \frac{3}{2}d$ the $n \times n$ matrix *is* the memory |
| The bandwidth measurement | The softmax runs at a fraction of peak FLOPs while saturating bandwidth. Attention's slowest step is the one doing the least arithmetic |
| Online softmax | Running $(m, \ell)$ with a retroactive $e^{m - m'}$ correction. **Exact**, one pass, never needs the whole row |
| Tiled attention | Blocks of $Q$ against blocks of $K,V$, correcting the output accumulator the same way. Peak intermediate memory independent of $n$ |
| Verification against the fused kernel | Agreement to float32 rounding, and FlashAttention approximates nothing |
| Recomputation in backward | Store $(m, \ell)$, recompute $S$. Spending FLOPs to save bytes is the right trade on this hardware |
| The race | We won the memory argument and lost the wall-clock one, because a schedule in Python is not a fused kernel |

**What this changes about everything before it.** FlashAttention is why the
$O(n^2)$ term stopped being an emergency. Context windows went from 2k to 128k+
without a new attention mechanism, because the constant in front of the
quadratic term fell through the floor, and every long-context Transformer you
have used depends on it.

It also reframes Chapter 15's comparison. The honest statement is not "quadratic
attention versus linear SSMs"; it is "quadratic attention *with an excellent
kernel* versus linear SSMs *with a good one*", and the crossover is much further
out than the asymptotics alone suggest. That is precisely why attention has not
been displaced, and why the models that do displace it in production are hybrids
that keep some of it.

**Where the curriculum has taken you.** A neuron trained with hand-derived
gradients, convolutions, residual connections, transfer learning. Autoencoders,
VAEs, GANs, diffusion. Attention from three matrix multiplies up through Vision
and Diffusion Transformers. Contrastive pretraining, and a language model that
can read a picture. Then three chapters on the frontier of what comes next:
architectures that escape the quadratic term, methods that fit a large model on
one GPU, and the kernel-level thinking that decides which of those actually
matters in practice.

Every claim in this chapter was measured rather than asserted, and across these
last five chapters several of those measurements disagreed with the headline.
That habit, building it, measuring it, and reporting what actually came out, is
the part worth keeping.
